# Grafos y enfermedades

En este notebook vamos a estudiar algunos de los grafos más típicos en la literatura. Después, vamos a analizar la estructura de una red de enfermedades construida a partir de datos de OMIM (Online Mendelian Inheritance in Man). En esta red, cada nodo representa una enfermedad y existe una conexión entre dos enfermedades cuando comparten al menos un gen asociado.

El objetivo es explorar cómo se organizan las enfermedades en función de su base genética, analizar patrones de agrupamiento entre distintas clases clínicas (neurológicas, cardiovasculares, etc.) y entender qué tipo de relaciones emergen a nivel global en la red. Para ello, utilizaremos herramientas de análisis de grafos y visualización que nos permitan identificar comunidades, nodos relevantes y posibles conexiones entre enfermedades aparentemente no relacionadas.

Más completa: https://exploring-data.com/vis/human-disease-network/#

In [ ]:
import networkx as nx
import igraph as ig
import numpy as np
from matplotlib import pyplot as plt
import requests
from matplotlib.lines import Line2D

### Construcción manual de un grafo

Construímos primero un grafo pequeño de forma manual, añadiendo explícitamente los nodos y las aristas.  
Es un ejemplo sencillo para introducir la estructura básica de un grafo: un conjunto de vértices conectados entre sí mediante enlaces.

In [ ]:
g = ig.Graph()
g.add_vertices([0,1,2,3,4])
g.add_edges([(0,1), (1,2), (2,3), (2,4)])
nx.draw(nx.from_edgelist(g.get_edgelist()))

### Grafo aleatorio de Erdős-Rényi

Aquí se genera un grafo aleatorio con 100 nodos, donde cada posible conexión entre dos nodos existe con probabilidad 0.1.  

Este modelo es útil como punto de partida para entender redes sin estructura compleja, en las que las conexiones aparecen de forma puramente aleatoria.

In [ ]:
er = ig.Graph.Erdos_Renyi(100, 0.1)
plt.figure(figsize=(10, 8))
nx.draw(nx.from_edgelist(er.get_edgelist()))
plt.show()

### Grafo de Barabási-Albert

Aquí se genera un grafo siguiendo el modelo de **Barabási-Albert**, basado en el mecanismo de *preferential attachment*.  

La idea es que los nuevos nodos tienden a conectarse con nodos que ya tienen muchas conexiones, lo que produce redes con unos pocos nodos muy conectados y muchos nodos con pocas conexiones.

In [ ]:
ba = ig.Graph.Barabasi(100)
plt.figure(figsize=(10, 8))
nx.draw(nx.from_edgelist(ba.get_edgelist()))
plt.show()

Para ver la distribución de nodos, creamos un grafo Barabási-Albert mucho más grande y se analiza su distribución de grados.  

Se representa en escala log-log la frecuencia de nodos con un determinado número de conexiones. Este tipo de visualización permite observar el comportamiento característico de muchas redes reales, donde aparecen pocos nodos muy conectados y muchos con grado bajo.

In [ ]:
from collections import Counter
ba_max = ig.Graph.Barabasi(100000)
cnt = Counter(ba_max.degree()).items()
deg_cnt = sorted(cnt)
degs, cnts = list(zip(*deg_cnt))
freq_cnts = np.array(cnts)/np.sum(cnts)
plt.plot(degs, freq_cnts)
plt.xscale('log', base=2)
plt.yscale('log', base=10)

## Grafo de enfermedades

### Carga del Human Disease Network (HDN)

Descargamos el grafo de enfermedades desde un archivo externo.  

En esta red, cada nodo representa una enfermedad y dos enfermedades están conectadas si comparten al menos un gen asociado. Después, el grafo se transforma en no dirigido para analizar únicamente la existencia de relación entre enfermedades, sin importar su dirección.

In [ ]:
url = "https://raw.githubusercontent.com/Jorgeprevi/ML_bootcamp_2026/refs/heads/main/grafos/disease_net.txt"

# Descargar el archivo
response = requests.get(url)
with open("disease_net.txt", "wb") as f:
    f.write(response.content)

# Leer con igraph
graph = ig.Graph.Read_Ncol("disease_net.txt", weights=True, names=True)
graph = graph.as_undirected(mode='collapse')

Calculamos algunas propiedades estructurales básicas del grafo:

- número de nodos y aristas,
- tamaño de la componente gigante,
- número de triángulos cerrados,
- diámetro de la componente principal,
- coeficiente de clustering.

Estas métricas permiten resumir cómo de conectada está la red y hasta qué punto presenta estructura local y organización global.

In [ ]:
print('Nodos: %d' % graph.vcount())
print('Vértices: %d' % graph.ecount())
c = graph.components(mode='strong')
giant = c.giant()
print('Componente gigante: %d' %  giant.vcount())
print('Triángulos cerrados: %d' % graph.as_directed().triad_census().t300)
print('Diámetro: %d' % giant.diameter())
print('Coeficiente de clustering: %3.3f' % graph.transitivity_avglocal_undirected())

Representamos el grafo completo usando un layout de tipo *spring*, que sitúa más cerca los nodos con mayor relación estructural.  

Esta visualización permite hacerse una primera idea de la forma global de la red, aunque en grafos grandes suele ser más útil para detectar zonas densas que para inspeccionar nodos concretos.

In [ ]:
nx_graph = nx.from_edgelist(graph.get_edgelist())
spring_pos = nx.spring_layout(nx_graph)
plt.figure(figsize=(10, 8))
nx.draw(nx_graph, pos=spring_pos, node_size=3, node_color='black', edge_color='black', width=1)
plt.show()

Asignamos las categorías o clases de enfermedades desde un archivo externo, y a cada vértice su clase correspondiente.  

Esto permite enriquecer la red con información semántica y después analizar si las enfermedades de una misma categoría tienden a agruparse o a compartir conexiones.

In [ ]:
# --- Cargar clases ---
url = "https://raw.githubusercontent.com/Jorgeprevi/ML_bootcamp_2026/refs/heads/main/grafos/disease_classes.txt"
response = requests.get(url)
response.raise_for_status()

lines = response.text.splitlines()
did2class = dict(line.strip().split('\t') for line in lines)

# Asignar clase a cada vértice del grafo de igraph
graph.vs["dclass"] = [did2class.get(did, "Unclassified") for did in graph.vs["name"]]

# Orden estable de clases
dclasses = sorted(set(graph.vs["dclass"]))

In [ ]:
# Paleta
palette = [
    'orchid', 'peru', 'pink', 'plum', 'powderblue', 'purple', 'red', 'rosybrown',
    'royalblue', 'orange', 'salmon', 'silver', 'skyblue', 'slategray', 'springgreen',
    'steelblue', 'tomato', 'turquoise', 'violet', 'wheat', 'lightgray', 'yellow'
]

class2color = {dclass: palette[i % len(palette)] for i, dclass in enumerate(dclasses)}

# --- Convertir a networkx preservando nombres ---
node_names = graph.vs["name"]
edges = [(node_names[i], node_names[j]) for i, j in graph.get_edgelist()]
nx_graph = nx.Graph()
nx_graph.add_edges_from(edges)

# Atributos por nodo
node_classes = {v["name"]: v["dclass"] for v in graph.vs}
nx.set_node_attributes(nx_graph, node_classes, "dclass")

node_colors = [class2color[nx_graph.nodes[n]["dclass"]] for n in nx_graph.nodes()]

# --- Layout mejorado ---
plt.figure(figsize=(16, 12))
pos = nx.spring_layout(
    nx_graph,
    k=0.35, 
    iterations=200,
    seed=42
)

# Dibujar edges primero, suaves
nx.draw_networkx_edges(
    nx_graph,
    pos,
    edge_color="gray",
    width=0.7,
    alpha=0.8
)

# Dibujar nodos
nx.draw_networkx_nodes(
    nx_graph,
    pos,
    node_color=node_colors,
    node_size=80,
    alpha=0.9,
    linewidths=0.3,
    edgecolors="black"
)

# Leyenda
legend_elements = [
    Line2D(
        [0], [0],
        marker='o',
        color='w',
        label=dclass,
        markerfacecolor=class2color[dclass],
        markeredgecolor='black',
        markersize=8
    )
    for dclass in dclasses
]

plt.legend(
    handles=legend_elements,
    title="Clase de enfermedad",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.title("Grafo de enfermedades por clase de enfermedad", fontsize=16)
plt.axis("off")
plt.tight_layout()
plt.show()

### Subgrafo de enfermedades oftalmológicas

Extraemos el subgrafo formado únicamente por las enfermedades de clase **Ophthamological**.  

Las enfermedades oftalmológicas aparecen menos conectadas porque suelen ser más específicas y afectan a estructuras muy concretas del ojo, lo que implica que comparten menos genes entre sí. Además, hay menos genes “hub” implicados en múltiples patologías oculares, por lo que la probabilidad de que dos enfermedades estén conectadas en la red es menor, dando lugar a una estructura más dispersa y fragmentada.

In [ ]:
eye_nodes = graph.vs.select(dclass='Ophthamological')
print('Eye-related diseases:', len(eye_nodes))

subgraph = graph.subgraph(eye_nodes)
labels = dict(enumerate(subgraph.vs['disease']))

nx_graph = nx.Graph()
nx_graph.add_nodes_from(labels.keys())
nx_graph.add_edges_from(subgraph.get_edgelist())

print('Nodes:', nx_graph.number_of_nodes(), 'Edges:', nx_graph.number_of_edges())

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(nx_graph, seed=42, k=0.8)
nx.draw(
    nx_graph,
    pos=pos,
    node_size=300,
    alpha=0.8,
    with_labels=True,
    labels=labels,
    font_size=8
)
plt.show()

Cargamos el subconjunto de enfermedades relacionadas con el cáncer.

Las enfermedades relacionadas con el cáncer están mucho más conectadas porque comparten mecanismos biológicos fundamentales, como la proliferación celular, la apoptosis o la reparación del ADN. Muchos genes están implicados en múltiples tipos de cáncer (alta pleiotropía), lo que genera una gran cantidad de enlaces entre enfermedades y produce una red más densa y altamente interconectada.

In [ ]:
cancer_nodes = graph.vs.select(dclass='Cancer')
print('Eye-related diseases:', len(cancer_nodes))

subgraph = graph.subgraph(cancer_nodes)
labels = dict(enumerate(subgraph.vs['disease']))

nx_graph = nx.Graph()
nx_graph.add_nodes_from(labels.keys())
nx_graph.add_edges_from(subgraph.get_edgelist())

print('Nodes:', nx_graph.number_of_nodes(), 'Edges:', nx_graph.number_of_edges())

plt.figure(figsize=(20, 15))
pos = nx.spring_layout(nx_graph, seed=42, k=0.8)
nx.draw(
    nx_graph,
    pos=pos,
    node_size=300,
    alpha=0.9,
    with_labels=True,
    labels=labels,
    font_size=8
)
plt.show()

Las enfermedades con un grado alto son aquellas que están conectadas con muchas otras dentro de la red. Esto suele indicar que comparten genes con múltiples patologías, lo que apunta a una base genética compleja o a la participación de mecanismos biológicos comunes. En la práctica, estas enfermedades funcionan como “hubs” del sistema, ya que concentran muchas conexiones y reflejan puntos clave donde convergen distintos procesos biológicos.

In [ ]:
deg_dis = sorted([(n.degree(), n['disease']) for n in graph.vs], reverse=True)
print('Max deg.:\n%s' %  '\n'.join('%d %s' % (deg, dis) for deg, dis in deg_dis[:10]))

Por otro lado, la betweenness mide qué enfermedades actúan como puente entre diferentes zonas de la red. Este tipo de enfermedades puede revelar vínculos inesperados entre categorías aparentemente separadas, sugiriendo que comparten mecanismos biológicos subyacentes aunque pertenezcan a dominios clínicos diferentes.

In [ ]:
nbs = graph.betweenness()
bs_dis = sorted([(bs, n['disease']) for bs, n in zip(nbs, graph.vs)], reverse=True)
print('Betweenness:\n%s' % '\n'.join('%4.3f %s' % (bs, dis) for bs, dis in bs_dis[:10]))